In [29]:
import os
from dotenv import load_dotenv

load_dotenv() # This loads the GROQ_API_KEY from your .env file

True

In [30]:
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load the EXACT SAME embedding model you used for building
# If you change this, your existing embeddings will not match your queries.
local_embedding_model_name = "intfloat/multilingual-e5-large"
emb = HuggingFaceEmbeddings(model_name=local_embedding_model_name)

# 2. Point to the existing persistent directory
persistent_directory = "D:/Final-Year-Project/data/vector_store"
client = chromadb.PersistentClient(path=persistent_directory)

# 3. Initialize Chroma WITHOUT adding documents
# Setting 'client' and 'collection_name' will automatically load the existing data.
vectorstore = Chroma(
    collection_name="self-correcting-RAG-Model", # Must match original name
    embedding_function=emb,
    client=client
)

# 4. Recreate your retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print("Successfully connected to existing vector store.")

Successfully connected to existing vector store.


In [31]:
# Use .invoke() instead of .get_relevant_documents()
docs = retriever.invoke("what are the four phase in validation of an indigenously developed diagnostic test?")

# Then you can print the results
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)
    print(doc.metadata)


--- Result 1 ---
The ICMR Standard Validation Protocol for COVID-19 testing involves four stages: 

1. Sample collection and transportation.
2. Laboratory testing and extraction of RNA.
3. Real-time Reverse Transcription Polymerase Chain Reaction (RT-PCR) for SARS-CoV-2 detection.
4. Result interpretation and reporting.
{'original': 'Standard Validation Protocol /ICMR', 'modality': 'text', 'id': 'ec539ec1-f9dc-4213-ad3b-0baa6150fbab'}

--- Result 2 ---
The ICMR 4 Standard Validation Protocol is a four-step process to validate COVID-19 tests. It includes:

1. Identification and selection of a validation site
2. Collection of clinical specimens and testing
3. Analysis of results and validation
4. Verification and validation of the test kit.
{'modality': 'text', 'original': 'Standard Validation Protocol /ICMR 4', 'id': '93155c47-5efc-4f1c-bdb9-e64afbfe2517'}

--- Result 3 ---
The Indian Council of Medical Research (ICMR) Standard Validation Protocol involves a 2-stage validation process:

In [32]:
def format_context(retrieved_docs, include_tables=True, include_images_as_text=True):
    """
    Make a compact text context. We’ll use image *summaries* as text by default.
    You can set include_images_as_text=False.
    """
    lines = []
    for i, d in enumerate(retrieved_docs, 1):
        m = d.metadata.get("modality", "text")
        if m == "text":
            # Keep summary + a short original snippet (optional)
            orig = d.metadata.get("original", "")
            snippet = orig[:1200]  # keep it short to save tokens
            #lines.append(f"[{i}] TEXT — Summary: {d.page_content}\nOriginal snippet: {snippet}")
            lines.append(f"[{i}] TEXT — Summary: {d.page_content}")
        elif m == "table" and include_tables:
            lines.append(f"[{i}] TABLE — Summary: {d.page_content}\n(HTML omitted for brevity)")
        elif m == "image" and include_images_as_text:
            lines.append(f"[{i}] IMAGE — Summary: {d.page_content}")
    return "\n\n".join(lines)


In [45]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq  # or your ChatGroq import

SYSTEM = """
 You are an expert assistant in the field of microbiology and diagnostics. Based on the provided context, 
 explain the ICMR Standard Validation Protocol for diagnostics.
 Note that there may be different stages or versions mentioned; 
 Always add from whicich line or document did you get the context ("i.e": according to the (Document name, page no.) it mentions that...),
 then the exact original lines from the document, and then your explaination.
 please explain the different processes described in the context.
 If asked to explain in detail, make sure to explain each and every step in detail.
 Never miss the important sub points form the documents, include every point in the answer.
 make sure to answer is in easy language without loosing the important words and numbers,
 Give a special attention note to the important topics which are necessary to be seen.
 If the answer is retrieved from the image or table, give the answer in the table format while their explainantion in sentence format
 for reference and tell them this is the image where this answer is been retrived
 """

PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    ("human", "Question: {question}\n\nContext:\n{context}")
])

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=4096,
    timeout=60, # Increase timeout to 60 seconds
    max_retries=2 # Try again automatically if it fails
)

def rag_answer(question: str):
    retrieved = retriever.invoke(question)
    # print("Retrieved Context", retrieved)
    ctx = format_context(retrieved, include_tables=True, include_images_as_text=True)
    chain = ({"question": RunnablePassthrough(), "context": RunnablePassthrough()}
             | PROMPT
             | llm
             | StrOutputParser())
    return chain.invoke({"question": question, "context": ctx}), retrieved



In [46]:
# Query
answer, sources = rag_answer(""" Explain the phases of development and validation of assays for new
rapid diagnostics in pathogen identification and antimicrobial
susceptibility testing (AST) in very detail""")
print(answer)
# print sources if you want:
for d in sources:
   print(d.metadata["modality"])

The Indian Council of Medical Research (ICMR) Standard Validation Protocol for diagnostics is a comprehensive framework for developing and validating new rapid diagnostic assays for identifying pathogens and conducting antimicrobial susceptibility testing (AST). The protocol is designed to ensure the accuracy, precision, and reliability of these tests, which are crucial for making informed clinical decisions.

**Phase 1: Assay Development**

The first phase of the protocol involves the development of a new rapid diagnostic assay. This phase includes:

1. **Conceptualization**: The development of a new assay begins with the conceptualization of a test that can identify a specific pathogen or detect antimicrobial resistance.
2. **Design and Development**: The assay is designed and developed using various techniques, such as molecular biology, immunology, or biochemistry.
3. **Pilot Testing**: The assay is pilot-tested to evaluate its performance and identify any potential issues.

**Phas